# Demand Forecasting and Inventory Optimization

This notebook forecasts daily demand for every store and product combination (SKU) and turns that forecast into an inventory policy: how much safety stock to hold and when to reorder.

**Workflow**

1. Load and check the sales data
2. Explore what drives demand
3. Build features, including lagged demand, so the model can learn from recent history
4. Split the timeline into training, calibration and test periods
5. Train an XGBoost model and compare it against simple baselines
6. Calculate safety stock and reorder points per SKU, using forecast error as the measure of uncertainty
7. Simulate a replenishment policy on the test period to check whether it actually hits the service level target
8. Summarize results for stakeholders

The notebook is designed to run from top to bottom. All assumptions are set once in the configuration cell below.

## 1. Setup and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

# Data location (the loader below also handles the original zip file or a manual upload)
DATA_PATH = "/content/sales_data.csv"

# Forecasting
FORECAST_HORIZON = 7        # days ahead the forecast is made; lag features only use data at least this old
TEST_SHARE = 0.20           # last 20% of dates are held out for final testing
CALIBRATION_SHARE = 0.10    # the 10% before that is used to measure forecast error for safety stock
INCLUDE_INVENTORY_LEVEL = False  # set True only if Inventory Level is recorded BEFORE the day's sales
RANDOM_STATE = 42

# Inventory policy
LEAD_TIME = 7               # days between placing and receiving an order
SERVICE_LEVEL = 0.95        # target probability of no stockout during a replenishment cycle
REVIEW_PERIOD = 1           # stock is checked once a day
ORDER_COVER_DAYS = 14       # each order tops stock up to cover this many days beyond the protection interval

# An order placed today protects stock until the order placed at the NEXT review arrives,
# so the reorder point must cover lead time plus the review period.
PROTECTION_DAYS = LEAD_TIME + REVIEW_PERIOD

KEYS = ["Store ID", "Product ID"]

assert FORECAST_HORIZON >= PROTECTION_DAYS - 1, (
    "The reorder decision needs forecasts for the whole protection interval, "
    "so FORECAST_HORIZON must be at least LEAD_TIME + REVIEW_PERIOD - 1."
)

## 2. Load and check the data

Each row is one product in one store on one day. Rows are sorted by date, store and product so that every SKU's history is in chronological order, which the lag features and the time based split both rely on.

In [ ]:
def load_sales_data(path):
    """Load sales_data.csv from DATA_PATH, the original zip file, or a Colab upload."""
    if os.path.exists(path):
        return pd.read_csv(path)

    zip_path = "/content/demand forecast.zip"
    if os.path.exists(zip_path):
        import zipfile
        with zipfile.ZipFile(zip_path) as z:
            members = [m for m in z.namelist() if m.endswith("sales_data.csv")]
            if members:
                z.extract(members[0], "/content/demand_data")
                return pd.read_csv(os.path.join("/content/demand_data", members[0]))

    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Could not find {path}. Set DATA_PATH to the location of sales_data.csv.")
    print("sales_data.csv not found. Please upload it.")
    uploaded = files.upload()
    return pd.read_csv(next(iter(uploaded)))


df_sales = load_sales_data(DATA_PATH)
df_sales["Date"] = pd.to_datetime(df_sales["Date"])
df_sales = df_sales.sort_values(["Date"] + KEYS, kind="mergesort").reset_index(drop=True)

n_dates = df_sales["Date"].nunique()
n_skus = df_sales.groupby(KEYS).ngroups
print(f"Rows: {len(df_sales):,}   Dates: {n_dates}   SKUs (store x product): {n_skus}")
print(f"Date range: {df_sales['Date'].min().date()} to {df_sales['Date'].max().date()}")
print(f"Missing values: {int(df_sales.isnull().sum().sum())}")
print(f"Duplicate rows: {int(df_sales.duplicated().sum())}")
print(f"Duplicate (date, store, product) keys: {int(df_sales.duplicated(['Date'] + KEYS).sum())}")

# Lag features assume every SKU has one row per day with no gaps
assert len(df_sales) == n_dates * n_skus, "Some SKU days are missing; lag features would be misaligned."

display(df_sales.head())
display(df_sales.describe(include="all"))

## 3. Exploratory analysis

These charts show the overall demand pattern and which factors appear to move demand, before any modeling.

In [ ]:
daily_totals = df_sales.groupby("Date")[["Demand", "Units Sold"]].sum()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily_totals.index, daily_totals["Demand"], label="Total demand")
ax.plot(daily_totals.index, daily_totals["Units Sold"], label="Total units sold", alpha=0.8)
ax.set_title("Total daily demand and units sold, all SKUs")
ax.set_xlabel("Date"); ax.set_ylabel("Units"); ax.legend()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_sales["Demand"], bins=40, kde=True, ax=axes[0]).set_title("Distribution of daily SKU demand")
sns.histplot(df_sales["Units Sold"], bins=40, kde=True, ax=axes[1]).set_title("Distribution of daily SKU units sold")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["Category", "Region"]:
    weekly = (df_sales.groupby([pd.Grouper(key="Date", freq="W"), col])["Demand"].mean().reset_index())
    plt.figure(figsize=(15, 5))
    sns.lineplot(data=weekly, x="Date", y="Demand", hue=col)
    plt.title(f"Average daily SKU demand by {col} (weekly mean)")
    plt.legend(title=col, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, col in zip(axes, ["Weather Condition", "Seasonality", "Promotion", "Epidemic"]):
    sns.boxplot(data=df_sales, x=col, y="Demand", ax=ax)
    ax.set_title(f"Demand by {col}")
plt.tight_layout(); plt.show()

print("Row counts per category value:")
for col in ["Category", "Region", "Weather Condition", "Seasonality"]:
    print(f"\n{df_sales[col].value_counts().to_string()}")

## 4. Feature engineering

**Calendar features** (month, day, day of week, day of year, week of year) let the model learn weekly and seasonal patterns. `Year` is not used: tree models cannot extrapolate to year values they never saw in training, and the lag features already capture changes in overall level.

**Lag features** are the most important addition. Recent demand is usually the strongest predictor of future demand. Every lag and rolling statistic is shifted by `FORECAST_HORIZON` days, so the model only uses information that would genuinely be available when the forecast is made. With a horizon of 7 days, the forecast for any day is made a full week ahead, which matches the replenishment lead time.

**Excluded columns.** `Units Sold` and `Units Ordered` are only known after the day happens, so using them would leak the answer. `Inventory Level` is excluded by default for the same reason: if it is recorded at the end of the day it already reflects that day's sales. Set `INCLUDE_INVENTORY_LEVEL = True` only if you have confirmed it is an opening balance.

**One hot encoding** turns each categorical column into yes or no columns so the model can use them.

In [ ]:
df_feat = df_sales.copy()

# Calendar features
df_feat["Month"] = df_feat["Date"].dt.month
df_feat["Day"] = df_feat["Date"].dt.day
df_feat["DayOfWeek"] = df_feat["Date"].dt.dayofweek          # Monday = 0
df_feat["DayOfYear"] = df_feat["Date"].dt.dayofyear
df_feat["WeekOfYear"] = df_feat["Date"].dt.isocalendar().week.astype(int)
df_feat["Year"] = df_feat["Date"].dt.year                    # only used by the comparison model below

# Lag and rolling features, per SKU, using only demand at least FORECAST_HORIZON days old
h = FORECAST_HORIZON
LAGS = sorted({h, h + 1, 2 * h, 21, 28} - set(range(h)))
demand_by_sku = df_feat.groupby(KEYS, sort=False)["Demand"]
for lag in LAGS:
    df_feat[f"demand_lag_{lag}"] = demand_by_sku.shift(lag)

base = demand_by_sku.shift(h)
base_by_sku = base.groupby([df_feat[k] for k in KEYS], sort=False)
df_feat["demand_roll_mean_7"] = base_by_sku.transform(lambda s: s.rolling(7).mean())
df_feat["demand_roll_mean_28"] = base_by_sku.transform(lambda s: s.rolling(28).mean())
df_feat["demand_roll_std_7"] = base_by_sku.transform(lambda s: s.rolling(7).std())
lag_cols = [f"demand_lag_{lag}" for lag in LAGS] + ["demand_roll_mean_7", "demand_roll_mean_28", "demand_roll_std_7"]

# One hot encoding (the original columns are kept for grouping by SKU later)
categorical_cols = ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]
dummies = pd.get_dummies(df_feat[categorical_cols], drop_first=True, dtype=int)
df_feat = pd.concat([df_feat, dummies], axis=1)

# The first days of each SKU have no lag history yet, so they are dropped
df_model = df_feat.dropna(subset=lag_cols).reset_index(drop=True)

calendar_cols = ["Month", "Day", "DayOfWeek", "DayOfYear", "WeekOfYear"]
known_in_advance = ["Price", "Discount", "Promotion", "Competitor Pricing", "Epidemic"]
feature_cols = known_in_advance + calendar_cols + lag_cols + list(dummies.columns)
if INCLUDE_INVENTORY_LEVEL:
    feature_cols.append("Inventory Level")

print(f"Rows dropped for missing lag history: {len(df_feat) - len(df_model):,}")
print(f"Modeling rows: {len(df_model):,}   Features: {len(feature_cols)}")
print(f"Lag features: {lag_cols}")
display(df_model[["Date"] + KEYS + ["Demand"] + lag_cols].head())

## 5. Time based split

The timeline is split by date, never randomly, so the model is always tested on days that come after everything it learned from.

* **Training:** fits the model
* **Calibration:** measures how wrong the forecast tends to be for each SKU; this becomes the uncertainty used in safety stock
* **Test:** final, untouched period used for accuracy metrics and the inventory simulation

Keeping calibration separate from test means the inventory policy is judged on days it was not tuned on.

In [ ]:
dates = np.sort(df_model["Date"].unique())
test_start = dates[int(len(dates) * (1 - TEST_SHARE))]
calib_start = dates[int(len(dates) * (1 - TEST_SHARE - CALIBRATION_SHARE))]

train_mask = df_model["Date"] < calib_start
calib_mask = (df_model["Date"] >= calib_start) & (df_model["Date"] < test_start)
test_mask = df_model["Date"] >= test_start

X_train, y_train = df_model.loc[train_mask, feature_cols], df_model.loc[train_mask, "Demand"]
X_calib, y_calib = df_model.loc[calib_mask, feature_cols], df_model.loc[calib_mask, "Demand"]
X_test, y_test = df_model.loc[test_mask, feature_cols], df_model.loc[test_mask, "Demand"]

for name, mask in [("Training", train_mask), ("Calibration", calib_mask), ("Test", test_mask)]:
    d = df_model.loc[mask, "Date"]
    print(f"{name:<12} {d.min().date()} to {d.max().date()}   rows: {mask.sum():,}")

## 6. Model training and evaluation

The XGBoost model is compared against three references:

* **Naive:** predict the demand seen `FORECAST_HORIZON` days earlier
* **Rolling mean:** predict the average of the most recent 28 available days
* **Previous version:** the original notebook's feature set (no lag features, with `Year` and `Inventory Level`), retrained on the same period

Metrics: MAE (average error in units), RMSE (penalizes large errors more), WAPE (total absolute error as a share of total demand, easy to read as a percentage), and R².

In [ ]:
def evaluate(actual, predicted):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE %": 100 * np.abs(actual - predicted).sum() / actual.sum(),
        "R2": r2_score(actual, predicted),
    }

xgb_params = dict(objective="reg:squarederror", n_estimators=100, learning_rate=0.1, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)

# Main model
model = xgb.XGBRegressor(**xgb_params)
model.fit(X_train, y_train)
test_pred = np.clip(model.predict(X_test), 0, None)

# Comparison: original feature set without lags
prev_cols = ["Inventory Level", "Year"] + known_in_advance + calendar_cols + list(dummies.columns)
prev_model = xgb.XGBRegressor(**xgb_params)
prev_model.fit(df_model.loc[train_mask, prev_cols], y_train)
prev_pred = prev_model.predict(df_model.loc[test_mask, prev_cols])

results = pd.DataFrame({
    f"Naive (demand {FORECAST_HORIZON} days earlier)": evaluate(y_test, df_model.loc[test_mask, f"demand_lag_{FORECAST_HORIZON}"]),
    "Rolling mean (28 days)": evaluate(y_test, df_model.loc[test_mask, "demand_roll_mean_28"]),
    "Previous version (no lags)": evaluate(y_test, prev_pred),
    "XGBoost with lag features": evaluate(y_test, test_pred),
}).T
display(results.round(2))

**Reading the comparison.** If the previous version scores noticeably better than the naive baseline mainly because of `Inventory Level`, that is a sign the column is leaking same day information. Check the feature importance chart below: a leaking feature typically dominates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (m, cols, title) in zip(axes, [(model, feature_cols, "Current model"),
                                       (prev_model, prev_cols, "Previous version")]):
    imp = pd.Series(m.feature_importances_, index=cols).sort_values(ascending=False).head(15)
    sns.barplot(x=imp.values, y=imp.index, ax=ax, color="steelblue")
    ax.set_title(f"Top 15 feature importances: {title}")
plt.tight_layout(); plt.show()

In [ ]:
test_df = df_model.loc[test_mask, ["Date"] + KEYS + ["Demand"]].copy()
test_df["Forecast"] = test_pred

totals = test_df.groupby("Date")[["Demand", "Forecast"]].sum()
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(totals.index, totals["Demand"], label="Actual demand", color="darkblue", linewidth=2)
ax.plot(totals.index, totals["Forecast"], label="Forecast", color="darkorange", linestyle="--", linewidth=2)
ax.set_title("Test period: total daily demand across all SKUs, actual vs forecast")
ax.set_xlabel("Date"); ax.set_ylabel("Units"); ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(test_df["Demand"], test_df["Forecast"], alpha=0.15, s=8)
lim = [0, max(test_df["Demand"].max(), test_df["Forecast"].max())]
ax.plot(lim, lim, color="red", linestyle="--", label="Perfect forecast")
ax.set_title("Test period: actual vs forecast, per SKU day")
ax.set_xlabel("Actual demand"); ax.set_ylabel("Forecast"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Inventory policy per SKU

Safety stock and reorder points are calculated **separately for every store and product**, because each SKU has its own demand level and variability.

The uncertainty used is the **forecast error** (actual minus forecast) measured on the calibration period, not the raw spread of demand. Once replenishment is planned from a forecast, the risk of a stockout comes from the forecast being wrong, not from demand varying in ways the forecast already anticipates.

**Protection interval.** Stock is reviewed once a day. If no order is placed today, the next order can only go in tomorrow and arrives a day later, so each decision has to protect against demand over the lead time **plus** the review period (`PROTECTION_DAYS` = `LEAD_TIME` + `REVIEW_PERIOD`, which is 8 days with the defaults). Ignoring the review period is a common reason real reorder policies miss their service level.

What matters is the error accumulated over that whole interval. The textbook shortcut σ(daily error) × √(lead time) assumes each day's error is independent of the next. In practice forecast errors tend to persist (a SKU running hot for a few days is common), which makes the shortcut too optimistic. So the notebook measures the lead time error directly: it sums the forecast errors over every rolling window of `PROTECTION_DAYS` days in the calibration period and takes the standard deviation of those sums.

* Safety stock = z × σ(forecast error summed over the protection interval)
* Reorder point = average daily forecast × protection interval + safety stock

Both versions of σ are shown in the table so the difference is visible.

In [ ]:
z = norm.ppf(SERVICE_LEVEL)

calib_df = df_model.loc[calib_mask, ["Date"] + KEYS + ["Demand"]].copy()
calib_df["Forecast"] = np.clip(model.predict(X_calib), 0, None)
calib_df["Error"] = calib_df["Demand"] - calib_df["Forecast"]

policy = test_df.groupby(KEYS).agg(avg_daily_forecast=("Forecast", "mean"),
                                   avg_daily_actual=("Demand", "mean"))
calib_df = calib_df.sort_values(["Date"] + KEYS)
error_by_sku = calib_df.groupby(KEYS, sort=False)["Error"]
calib_df["ProtectionError"] = error_by_sku.transform(lambda s: s.rolling(PROTECTION_DAYS).sum())

policy["sigma_daily_error"] = error_by_sku.std()
policy["sigma_raw_demand"] = calib_df.groupby(KEYS)["Demand"].std()                            # comparison only
policy["sigma_protection_if_independent"] = policy["sigma_daily_error"] * np.sqrt(PROTECTION_DAYS)  # comparison only
policy["sigma_protection_error"] = calib_df.groupby(KEYS)["ProtectionError"].std()
policy["safety_stock"] = z * policy["sigma_protection_error"]
policy["reorder_point"] = policy["avg_daily_forecast"] * PROTECTION_DAYS + policy["safety_stock"]

print(f"Service level {SERVICE_LEVEL:.0%} gives z = {z:.3f}; lead time {LEAD_TIME} days "
      f"+ review period {REVIEW_PERIOD} = protection interval {PROTECTION_DAYS} days")
print(f"Median daily sigma: forecast error {policy['sigma_daily_error'].median():.1f} "
      f"vs raw demand {policy['sigma_raw_demand'].median():.1f} units")
print(f"Median protection interval sigma: measured {policy['sigma_protection_error'].median():.1f} "
      f"vs independence shortcut {policy['sigma_protection_if_independent'].median():.1f} units")
error_autocorr = error_by_sku.apply(lambda s: s.autocorr(1)).median()
print(f"Median day to day correlation of forecast errors: {error_autocorr:.2f}")
display(policy.round(1).sort_values("avg_daily_forecast", ascending=False))

## 8. Replenishment simulation

This replays the test period day by day for each SKU to check whether the policy works:

1. At the start of each day, any order due that day arrives.
2. Actual demand is served from stock on hand; anything that cannot be served is a lost sale.
3. At the end of the day, if stock on hand plus stock already on order is at or below the reorder point, a new order is placed. It arrives after `LEAD_TIME` days.

The reorder point here is dynamic: it uses the forecast for the specific days in the upcoming protection interval rather than a flat average, plus the SKU's safety stock. Each order tops the inventory position up to cover the protection interval plus `ORDER_COVER_DAYS`.

A 95% service level means that roughly 95% of replenishment cycles (the lead time after each order) should finish without a stockout.

In [ ]:
def simulate_sku(sku_df, safety_stock, lead_time=LEAD_TIME, protection=PROTECTION_DAYS, cover_days=ORDER_COVER_DAYS):
    d = sku_df.sort_values("Date").reset_index(drop=True)
    demand = d["Demand"].to_numpy(float)
    fc = d["Forecast"].to_numpy(float)
    n = len(d)
    # Extend the forecast past the end of the test period so lookahead windows stay full
    fc_ext = np.concatenate([fc, np.full(protection + cover_days + 1, fc[-28:].mean())])

    def fc_sum(start, days):
        return fc_ext[start:start + days].sum()

    on_hand = fc_sum(0, protection + cover_days) + safety_stock
    arrivals = np.zeros(n + lead_time + 2)
    rec = {k: np.zeros(n) for k in ["on_hand_start", "on_hand_end", "sold", "lost", "rop", "order_qty"]}

    for t in range(n):
        on_hand += arrivals[t]
        rec["on_hand_start"][t] = on_hand
        sold = min(on_hand, demand[t])
        on_hand -= sold
        rec["sold"][t], rec["lost"][t], rec["on_hand_end"][t] = sold, demand[t] - sold, on_hand

        on_order = arrivals[t + 1:].sum()
        rop = fc_sum(t + 1, protection) + safety_stock
        rec["rop"][t] = rop
        if on_hand + on_order <= rop:
            order_up_to = fc_sum(t + 1, protection + cover_days) + safety_stock
            qty = order_up_to - (on_hand + on_order)
            arrivals[t + 1 + lead_time] += qty
            rec["order_qty"][t] = qty

    out = d[["Date", "Demand", "Forecast"]].copy()
    for k, v in rec.items():
        out[k] = v

    # A cycle is the lead time after an order; it succeeds if no day in it runs out
    stockout_days = out["lost"].to_numpy() > 1e-9
    order_days = np.flatnonzero(out["order_qty"].to_numpy() > 0)
    cycles = [not stockout_days[t + 1:t + 1 + lead_time].any() for t in order_days if t + lead_time < n]
    summary = {
        "orders": len(order_days),
        "cycle_service_level": np.mean(cycles) if cycles else np.nan,
        "fill_rate": out["sold"].sum() / out["Demand"].sum(),
        "stockout_days": int(stockout_days.sum()),
        "avg_on_hand": out["on_hand_end"].mean(),
        "lost_units": out["lost"].sum(),
    }
    return out, summary

In [ ]:
# Example SKU: the one with the highest average demand
example_key = policy["avg_daily_forecast"].idxmax()
example_df = test_df[(test_df[KEYS] == list(example_key)).all(axis=1)]
example_ss = policy.loc[example_key, "safety_stock"]
sim, sim_summary = simulate_sku(example_df, example_ss)

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax = axes[0]
ax.step(sim["Date"], sim["on_hand_start"], where="post", color="green", linewidth=2, label="Stock on hand (start of day)")
ax.plot(sim["Date"], sim["rop"], color="red", linestyle="--", label="Reorder point (dynamic)")
ax.axhline(example_ss, color="orange", linestyle=":", label=f"Safety stock ({example_ss:.0f})")
orders = sim[sim["order_qty"] > 0]
ax.scatter(orders["Date"], orders["on_hand_end"], marker="^", color="black", zorder=5, label="Order placed")
stockouts = sim[sim["lost"] > 0]
ax.scatter(stockouts["Date"], np.zeros(len(stockouts)), marker="x", color="red", s=60, zorder=5, label="Stockout")
ax.set_title(f"Simulated inventory for store {example_key[0]}, product {example_key[1]}")
ax.set_ylabel("Units"); ax.legend(loc="upper right")

axes[1].plot(sim["Date"], sim["Demand"], label="Actual demand", color="darkblue")
axes[1].plot(sim["Date"], sim["Forecast"], label="Forecast", color="darkorange", linestyle="--")
axes[1].set_ylabel("Units per day"); axes[1].set_xlabel("Date"); axes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

print({k: round(v, 3) if isinstance(v, float) else v for k, v in sim_summary.items()})

In [ ]:
# Run the simulation for every SKU
sim_rows = []
for key, grp in test_df.groupby(KEYS):
    _, s = simulate_sku(grp, policy.loc[key, "safety_stock"])
    sim_rows.append({"Store ID": key[0], "Product ID": key[1], **s})
sim_results = pd.DataFrame(sim_rows).set_index(KEYS)
policy = policy.join(sim_results)

overall_fill = 1 - sim_results["lost_units"].sum() / test_df["Demand"].sum()

print(f"Target cycle service level:        {SERVICE_LEVEL:.1%}")
print(f"Achieved cycle service level:      {sim_results['cycle_service_level'].mean():.1%} (average across SKUs)")
print(f"Overall fill rate (units served):  {overall_fill:.1%}")
print(f"SKUs below target:                 {(sim_results['cycle_service_level'] < SERVICE_LEVEL).sum()} of {len(sim_results)}")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(sim_results["cycle_service_level"], bins=20, ax=ax)
ax.axvline(SERVICE_LEVEL, color="red", linestyle="--", label="Target")
ax.set_title("Achieved cycle service level per SKU"); ax.set_xlabel("Share of cycles without stockout"); ax.legend()
plt.tight_layout(); plt.show()

display(policy.round(2).sort_values("cycle_service_level"))

## 9. Stakeholder summary

The summary below is generated from the results above, so the numbers update automatically whenever the notebook is rerun.

In [ ]:
best = results.loc["XGBoost with lag features"]
naive = results.iloc[0]
prev = results.loc["Previous version (no lags)"]
ex = policy.loc[example_key]
top_drivers = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(4).index
sigma_measured = policy["sigma_protection_error"].median()
sigma_shortcut = policy["sigma_protection_if_independent"].median()
lags_help = best["WAPE %"] < prev["WAPE %"] - 1

lag_line = (
    f"Adding lag features improved WAPE from {prev['WAPE %']:.1f}% (previous version) to {best['WAPE %']:.1f}%."
    if lags_help else
    f"Lag features add little here (previous version WAPE {prev['WAPE %']:.1f}%): demand in this dataset depends only "
    f"weakly on its own recent history, so accuracy comes mainly from {', '.join(top_drivers)}."
)
sigma_line = (
    f"Error over the {PROTECTION_DAYS} day protection interval is measured directly (median σ **{sigma_measured:.0f} units**) "
    + (f"because errors persist from day to day; the independence shortcut would understate it at **{sigma_shortcut:.0f}**."
       if sigma_measured > 1.15 * sigma_shortcut else
       f"and is close to the independence shortcut (**{sigma_shortcut:.0f}**), so daily errors are roughly independent here.")
)

Markdown(f"""
### Forecast accuracy (test period {test_df['Date'].min().date()} to {test_df['Date'].max().date()})

* Forecasts are made **{FORECAST_HORIZON} days ahead** for each of **{len(policy)} SKUs**.
* Typical error per SKU per day: **{best['MAE']:.1f} units** (WAPE **{best['WAPE %']:.1f}%**, R² **{best['R2']:.2f}**).
* Versus the naive baseline: WAPE {naive['WAPE %']:.1f}% for naive, so the model {'reduces' if best['WAPE %'] < naive['WAPE %'] else 'does not reduce'} error by **{naive['WAPE %'] - best['WAPE %']:.1f} points**.
* {lag_line}

### Inventory policy

* Safety stock and reorder points are set per SKU at a **{SERVICE_LEVEL:.0%} service level** with a **{LEAD_TIME} day lead time**, reviewed daily ({PROTECTION_DAYS} day protection interval).
* Uncertainty is based on forecast error rather than raw demand variability: median daily σ of **{policy['sigma_daily_error'].median():.1f} units** versus **{policy['sigma_raw_demand'].median():.1f}** for raw demand.
* {sigma_line}
* Example, store {example_key[0]} product {example_key[1]} (highest volume): safety stock **{ex['safety_stock']:.0f} units**, average reorder point **{ex['reorder_point']:.0f} units**.

### Does the policy work?

* Replaying the test period, **{sim_results['cycle_service_level'].mean():.1%}** of replenishment cycles finished without a stockout (target {SERVICE_LEVEL:.0%}).
* **{overall_fill:.1%}** of all demand units were served from stock.
* {(sim_results['cycle_service_level'] < SERVICE_LEVEL).sum()} of {len(sim_results)} SKUs fell below target. Each SKU has only about {sim_results['orders'].median():.0f} replenishment cycles in the test period, so a single stockout can push one SKU under 95%; the aggregate figure is the more reliable read, and SKUs that miss repeatedly are the ones worth reviewing.

### Assumptions and limitations

* Lead time and service level are assumptions set in the configuration cell, not values from the data.
* Weather is used as a feature as if it were known in advance; in practice a weather forecast would be used.
* The simulation assumes unmet demand is lost rather than backordered, and ignores order minimums, costs and shelf capacity.
""")